# `installer` 02: noteworthy single-feature findings

**Purpose:** identify and discuss supported points that stand out after the
standard `high-cardinality-category` breakdown. Target relationships here are
exploratory and must be rechecked after the split is frozen.


In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


def find_stage_directory():
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (
            (candidate / "data" / "TrainingSetValues.csv").exists()
            and (candidate / "src" / "source_data_validation.py").exists()
        ):
            return candidate
    raise FileNotFoundError("Could not locate the stage-1-pump-it-up directory.")


stage_directory = find_stage_directory()
source_directory = str((stage_directory / "src").resolve())
if source_directory not in sys.path:
    sys.path.insert(0, source_directory)

from predictor_audit import (
    analysis_categories,
    categorical_summary,
    categorical_target_profile,
    category_frequency_table,
    numeric_summary,
    numeric_target_summary,
    related_feature_summary,
    sentinel_mask,
    source_blank_mask,
    text_normalisation_summary,
)
from source_data_validation import (
    validate_aligned_ids,
    validate_label_frame,
    validate_raw_feature_schema,
)

data_directory = stage_directory / "data"
training_features = pd.read_csv(
    data_directory / "TrainingSetValues.csv",
    keep_default_na=False,
)
training_labels = pd.read_csv(
    data_directory / "TrainingSetLabels.csv",
    keep_default_na=False,
)
test_features = pd.read_csv(
    data_directory / "TestSetValues.csv",
    keep_default_na=False,
)

validate_raw_feature_schema(training_features)
validate_raw_feature_schema(test_features)
validate_label_frame(training_labels)
validate_aligned_ids(training_features, training_labels)

training_data = training_features.merge(
    training_labels,
    on="id",
    validate="one_to_one",
)

feature = 'installer'
feature_metadata = {'order': 5, 'name': 'installer', 'audit_type': 'high-cardinality-category', 'role': 'candidate', 'disposition': 'retain after conservative normalisation and fold-fitted rare grouping', 'finding': 'The field contains case and whitespace fragmentation as well as blank and sentinel values.', 'decision': 'Normalise case and whitespace, retain separately from funder and map unseen values explicitly.', 'risk': 'Aliases need a reviewed mapping; fuzzy merging can erase meaningful distinctions.', 'sentinel_tokens': ['0', 'unknown', 'not known', '-', 'unknown installer'], 'related': [{'feature': 'funder', 'reason': 'Funding and installation organisations are strongly related but not equivalent.'}, {'feature': 'construction_year', 'reason': 'Installers operate in particular construction cohorts.'}, {'feature': 'scheme_name', 'reason': 'Installers can be associated with repeated named schemes.'}]}
feature_types = {'amount_tsh': 'numeric', 'date_recorded': 'date', 'funder': 'high-cardinality-category', 'gps_height': 'numeric', 'installer': 'high-cardinality-category', 'longitude': 'coordinate', 'latitude': 'coordinate', 'wpt_name': 'high-cardinality-category', 'num_private': 'numeric', 'basin': 'category', 'subvillage': 'high-cardinality-category', 'region': 'category', 'region_code': 'category', 'district_code': 'category', 'lga': 'category', 'ward': 'high-cardinality-category', 'population': 'numeric', 'public_meeting': 'binary', 'recorded_by': 'constant', 'scheme_management': 'category', 'scheme_name': 'high-cardinality-category', 'permit': 'binary', 'construction_year': 'year', 'extraction_type': 'category', 'extraction_type_group': 'category', 'extraction_type_class': 'category', 'management': 'category', 'management_group': 'category', 'payment': 'category', 'payment_type': 'category', 'water_quality': 'category', 'quality_group': 'category', 'quantity': 'category', 'quantity_group': 'category', 'source': 'category', 'source_type': 'category', 'source_class': 'category', 'waterpoint_type': 'category', 'waterpoint_type_group': 'category'}
assert feature in training_features.columns
print(
    f"Validated {len(training_features):,} training rows and "
    f"{len(test_features):,} test rows for {feature}."
)


Validated 59,400 training rows and 14,850 test rows for installer.


## Supported target evidence


In [2]:
sentinel_tokens = ['0', 'unknown', 'not known', '-', 'unknown installer']
target_profile = categorical_target_profile(
    training_data,
    feature,
    minimum_support=100,
    sentinel_tokens=sentinel_tokens,
)
display(target_profile.head(20))

supported = target_profile.loc[target_profile["meets support threshold"]].copy()
non_functional_column = "non functional (%)"
if non_functional_column in supported:
    display(
        supported.sort_values(non_functional_column, ascending=False)
        .head(12)[["rows", non_functional_column]]
    )


status_group,rows,meets support threshold,functional (%),functional needs repair (%),non functional (%)
installer,,,,,
dwe,17405,True,54.20,9.32,36.48
<missing/blank>,3655,True,54.72,12.04,33.24
government,1891,True,29.19,13.64,57.17
hesawa,1395,True,56.34,3.87,39.78
rwe,1206,True,25.21,11.36,63.43
commu,1065,True,68.17,3.00,28.83
danida,1050,True,51.62,7.90,40.48
district council,965,True,40.10,6.32,53.58
kkkt,910,True,46.70,6.81,46.48


status_group,rows,non functional (%)
installer,,
fw,173,92.49
fini water,389,84.32
district water department,167,80.84
centr,162,80.25
halmashauri ya wilaya sikonge,102,78.43
central govt,138,78.26
finw,208,75.96
wizara ya maji,103,70.87
central government,763,70.38


## Observation

The field contains case and whitespace fragmentation as well as blank and sentinel values.

## Interpretation

The supported single-feature patterns make this field worth the stated
treatment, but they do not prove causation or independent predictive value.
High-cardinality and geographic fields are especially vulnerable to
memorisation under a random split.

## Provisional decision

Normalise case and whitespace, retain separately from funder and map unseen values explicitly.

**Risk to carry forward:** Aliases need a reviewed mapping; fuzzy merging can erase meaningful distinctions.


In [3]:
decision_record = pd.DataFrame([{
    "feature": feature,
    "role": feature_metadata["role"],
    "disposition": feature_metadata["disposition"],
    "finding": feature_metadata["finding"],
    "decision": feature_metadata["decision"],
    "risk": feature_metadata["risk"],
}])
display(decision_record.set_index("feature"))


,role,disposition,finding,decision,risk
feature,,,,,
installer,candidate,retain after conservative normalisation and fo...,The field contains case and whitespace fragmen...,"Normalise case and whitespace, retain separate...",Aliases need a reviewed mapping; fuzzy merging...
